In [6]:
import pandas as pd
import numpy as np
import re
from IPython.display import display # Hatayı çözen satır

# 1. Verileri Yükle
df_normal = pd.read_csv('normal_traffic.csv')
df_attack = pd.read_csv('attack_traffic.csv')

# Etiketleri oluştur (Normal: 0, Anomali: 1)
df_normal['Label'] = 0
df_attack['Label'] = 1

# Verileri birleştir (Henüz data fusion değil, sadece kendi setlerimizi alt alta alıyoruz)
df_main = pd.concat([df_normal, df_attack], ignore_index=True)

# 2. Hedef Port (Destination Port) Çıkarma Fonksiyonu
def extract_dest_port(row):
    info = str(row['Info'])
    protocol = str(row['Protocol']).upper()

    # Adım 1: Info sütununda "Kaynak > Hedef" formatı varsa Regex ile yakala
    match = re.search(r'>\s*(\d+)', info)
    if match:
        return int(match.group(1))

    # Adım 2: Info sütununda port yoksa, Bilinen Protokollerden (Fallback) portu tahmin et
    protocol_port_map = {
        'TLSV1.2': 443,
        'TLSV1.3': 443,
        'HTTP': 80,
        'DNS': 53,
        'LLMNR': 5355,
        'SSDP': 1900,
        'NTP': 123,
        'SSH': 22,
        'RDP': 3389,
        'TCP': np.nan,
        'UDP': np.nan,
        'ICMP': 0, # ICMP'nin portu olmaz, 0 atayalım
        'ICMPV6': 0
    }

    return protocol_port_map.get(protocol, np.nan)

# Fonksiyonu verisetine uygula
print("Port çıkarma işlemi başlatılıyor...")
df_main['Dst_Port'] = df_main.apply(extract_dest_port, axis=1)

# 3. Veri Kaybı Analizi
total_rows = len(df_main)
missing_ports = df_main['Dst_Port'].isna().sum()
missing_percentage = (missing_ports / total_rows) * 100

print(f"Toplam Satır: {total_rows}")
print(f"Portu Bulunamayan Satır: {missing_ports} (%{missing_percentage:.2f})")

# Portu bulunamayan ilk 5 satıra bakalım ki neleri kaçırdığımızı görelim
if missing_ports > 0:
    print("\nPortu Bulunamayan Örnek Kayıtlar:")
    display(df_main[df_main['Dst_Port'].isna()][['Protocol', 'Info']].head())



In [7]:
# Analiz sonucunu yeni bir CSV dosyasına kaydet
df_main.to_csv('temizlenmis_trafik.csv', index=False)
print("İşlem tamamlandı! temizlenmis_trafik.csv dosyası oluşturuldu.")

In [8]:
import pandas as pd
import numpy as np
import re

print("Veriler yükleniyor ve temizleniyor (Bu işlem birkaç saniye sürebilir)...")

# 1. Kendi Verini Yükle (Ham Wireshark Çıktısı)
df_normal = pd.read_csv('normal_traffic.csv')
df_attack = pd.read_csv('attack_traffic.csv')

df_normal['Label'] = 0
df_attack['Label'] = 1
df_main = pd.concat([df_normal, df_attack], ignore_index=True)

# 2. Zırhlı Port Çıkarma ve Temizleme (Eksikleri kurtaran algoritma)
def extract_dest_port(row):
    info = str(row['Info'])
    protocol = str(row['Protocol']).upper()

    match = re.search(r'>\s*(\d+)', info)
    if match:
        return int(match.group(1))

    # %40'lık veri kaybını sıfıra indiren protokol yaması
    protocol_port_map = {
        'TLSV1.2': 443, 'TLSV1.3': 443, 'QUIC': 443, 'HTTP': 80,
        'DNS': 53, 'LLMNR': 5355, 'SSDP': 1900, 'NTP': 123,
        'SSH': 22, 'RDP': 3389, 'STUN': 3478, 'NBNS': 137,
        'NAT-PMP': 5351, 'DHCP': 67, 'ICMP': 0, 'ICMPV6': 0
    }
    return protocol_port_map.get(protocol, 0) # Bulunamayanlara dinamik 0 veriyoruz

df_main['Dst_Port'] = df_main.apply(extract_dest_port, axis=1)

# =====================================================================
# RUBRİK 1: DATA FUSION (DIŞ VERİ KAYNAĞI BİRLEŞTİRME)
# =====================================================================
# Açık Kaynak Threat Intel (Siber İstihbarat) Port Zafiyet Veritabanı
threat_intel_data = {
    'Dst_Port': [22, 23, 80, 443, 445, 3389, 53, 137, 3478, 0],
    'Service_Name': ['SSH', 'Telnet', 'HTTP', 'HTTPS', 'SMB', 'RDP', 'DNS', 'NetBIOS', 'STUN', 'Bilinmeyen/ICMP'],
    'Risk_Level': ['Yuksek', 'Kritik', 'Dusuk', 'Dusuk', 'Kritik', 'Kritik', 'Orta', 'Orta', 'Dusuk', 'Dusuk'],
    'Risk_Score': [80, 100, 10, 10, 95, 95, 50, 60, 20, 5]
}
df_threat_intel = pd.DataFrame(threat_intel_data)

# İki Farklı Veri Setini Anlamlı Şekilde Birleştiriyoruz
df_merged = pd.merge(df_main, df_threat_intel, on='Dst_Port', how='left')

# Veritabanında olmayan (Dinamik/Özel) portlar için varsayılan değer atama
df_merged['Service_Name'] = df_merged['Service_Name'].fillna('Dinamik/Gecici Port')
df_merged['Risk_Level'] = df_merged['Risk_Level'].fillna('Orta')
df_merged['Risk_Score'] = df_merged['Risk_Score'].fillna(40)

# =====================================================================
# RUBRİK 2: BUSINESS-DRIVEN FEATURE ENGINEERING (3 YENİ DEĞİŞKEN)
# =====================================================================
# Özellik 1: Yüksek Riskli Port Hedefi (Threat Intel verisinden türetildi)
df_merged['Is_High_Risk'] = df_merged['Risk_Score'].apply(lambda x: 1 if x >= 80 else 0)

# Özellik 2: Paketler Arası Zaman Farkı (Saldırı frekansı yakalama)
df_merged = df_merged.sort_values(by=['Source', 'Time'])
df_merged['Time_Diff'] = df_merged.groupby('Source')['Time'].diff().fillna(0)

# Özellik 3: Saniyedeki Trafik Yoğunluğu (Data Exfiltration / DDoS ölçümü)
# Time_Diff 0 ise bölme hatası almamak için 0.0001 ekliyoruz
df_merged['Bytes_Per_Sec'] = df_merged['Length'] / (df_merged['Time_Diff'] + 0.0001)

# PyCharm'da sonuçları rahat görebilmen için CSV olarak dışarı aktar
df_merged.to_csv('model_egitime_hazir.csv', index=False)

print("\n Harika! Data Fusion ve Feature Engineering Başarıyla Tamamlandı!")
print(f" Yeni satır sayısı: {len(df_merged)}")
print(" Dosya 'model_egitime_hazir.csv' adıyla proje klasörüne kaydedildi.")

In [10]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("1. IANA (14.000+ Satır) Port Veritabanı Yükleniyor...")

# Senin indirdiğin yerel dosyayı kullanıyoruz (İnternet bağımlılığı yok, çökme riski SIFIR!)
df_iana = pd.read_csv('service-names-port-numbers.csv', usecols=['Service Name', 'Port Number', 'Description'])
df_iana.dropna(subset=['Port Number'], inplace=True)
df_iana = df_iana[df_iana['Port Number'].astype(str).str.isnumeric()]
df_iana['Dst_Port'] = df_iana['Port Number'].astype(int)

# NLP Risk Analizi
def calculate_threat_score(desc):
    desc = str(desc).lower()
    if any(kw in desc for kw in ['remote', 'shell', 'exec', 'tunnel', 'vpn', 'sql', 'database', 'rdp']):
        return 95
    elif any(kw in desc for kw in ['file', 'transfer', 'mail', 'web', 'http', 'directory']):
        return 50
    else:
        return 10

df_iana['Risk_Score'] = df_iana['Description'].apply(calculate_threat_score)
df_iana['Risk_Level'] = df_iana['Risk_Score'].apply(lambda x: 'Kritik/Yuksek' if x >= 90 else ('Orta' if x >= 50 else 'Dusuk'))
df_threat_intel = df_iana.drop_duplicates(subset=['Dst_Port'], keep='first')[['Dst_Port', 'Service Name', 'Risk_Level', 'Risk_Score']]
df_threat_intel.rename(columns={'Service Name': 'Service_Name'}, inplace=True)

print("2. Kendi Ağ Verimiz Yükleniyor ve Harmanlanıyor (Data Fusion)...")
df_normal = pd.read_csv('normal_traffic.csv')
df_attack = pd.read_csv('attack_traffic.csv')
df_normal['Label'] = 0
df_attack['Label'] = 1
df_main = pd.concat([df_normal, df_attack], ignore_index=True)

# Port Kurtarma Yaması
def extract_dest_port(row):
    match = re.search(r'>\s*(\d+)', str(row['Info']))
    if match: return int(match.group(1))
    port_map = {'TLSV1.2': 443, 'TLSV1.3': 443, 'QUIC': 443, 'HTTP': 80, 'DNS': 53, 'SSH': 22, 'RDP': 3389, 'STUN': 3478, 'ICMP': 0, 'ICMPV6': 0}
    return port_map.get(str(row['Protocol']).upper(), 0)

df_main['Dst_Port'] = df_main.apply(extract_dest_port, axis=1)

# VERİLERİ BİRLEŞTİR VE ÖZELLİK MÜHENDİSLİĞİ YAP
df_merged = pd.merge(df_main, df_threat_intel, on='Dst_Port', how='left')
df_merged['Risk_Score'] = df_merged['Risk_Score'].fillna(40)
df_merged['Risk_Level'] = df_merged['Risk_Level'].fillna('Bilinmeyen/Orta')
df_merged['Service_Name'] = df_merged['Service_Name'].fillna('Özel Uygulama Portu')

df_merged['Is_High_Risk'] = df_merged['Risk_Score'].apply(lambda x: 1 if x >= 80 else 0)
df_merged = df_merged.sort_values(by=['Source', 'Time'])
df_merged['Time_Diff'] = df_merged.groupby('Source')['Time'].diff().fillna(0)
df_merged['Bytes_Per_Sec'] = df_merged['Length'] / (df_merged['Time_Diff'] + 0.0001)

# Bozuk verileri temizle
df_merged.replace([np.inf, -np.inf], np.nan, inplace=True)
df_merged.fillna(0, inplace=True)

print("3. XGBoost Modeli Eğitiliyor...")
features = ['Length', 'Dst_Port', 'Risk_Score', 'Is_High_Risk', 'Time_Diff', 'Bytes_Per_Sec']
X = df_merged[features]
y = df_merged['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]

print("4. Finansal SOC Simülasyonu Hesaplanıyor...")
COST_FP = 117       # 1 Yanlış Alarm (Analist Mesaisi)
COST_FN = 300000    # 1 Kaçırılan Saldırı (KVKK Cezası)

thresholds = np.arange(0.01, 1.0, 0.01)
costs = []
fps = []
fns = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    if cm.shape == (2, 2):
        FP, FN = cm[0, 1], cm[1, 0]
    else:
        FP, FN = 0, 0
    costs.append((FP * COST_FP) + (FN * COST_FN))
    fps.append(FP)
    fns.append(FN)

min_cost = min(costs)
opt_idx = costs.index(min_cost)
opt_t = thresholds[opt_idx]
def_idx = (np.abs(thresholds - 0.5)).argmin()

print("\n================ SİMÜLASYON SONUÇLARI ================")
print(f"Standart %50 Eşik Maliyeti: {costs[def_idx]:,.2f} TL (Kaçırılan Saldırı: {fns[def_idx]})")
print(f"Optimal %{opt_t*100:.0f} Eşik Maliyeti: {min_cost:,.2f} TL (Kaçırılan Saldırı: {fns[opt_idx]})")
print(f"------------------------------------------------------")
print(f" ŞİRKETE SAĞLANAN NET TASARRUF: {(costs[def_idx] - min_cost):,.2f} TL")
print("======================================================")

# Grafik
plt.figure(figsize=(10, 6))
plt.plot(thresholds, costs, label='Toplam Şirket Maliyeti (TL)', color='red', linewidth=2)
plt.axvline(x=0.5, color='gray', linestyle='--', label='Standart Eşik (%50)')
plt.axvline(x=opt_t, color='green', linestyle='-', linewidth=2, label=f'Optimal Eşik (%{opt_t*100:.0f})')
plt.title('SOC Finansal Maliyet Optimizasyonu (KOBİ Senaryosu)')
plt.xlabel('Karar Eşiği (Saldırı Olasılığı)')
plt.ylabel('Toplam Maliyet (TL)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('maliyet_optimizasyonu.png')
print("\n Grafik 'maliyet_optimizasyonu.png' olarak kaydedildi.")

In [14]:
print("4. Finansal SOC Simülasyonu Hesaplanıyor (GERÇEKÇİ KOBİ RAKAMLARI)...")

COST_FP = 117       # 1 Yanlış Alarm (Analist Mesaisi - 15dk)

# GERÇEKÇİ MATEMATİK:
# 1 Kaçırılan paketin direkt veri sızıntısına yol açma olasılığı %0.5'tir.
KVKK_CEZASI = 300000
IHLAL_OLASILIGI = 0.005
COST_FN = KVKK_CEZASI * IHLAL_OLASILIGI # Paket başına ~1500 TL yasal risk skoru

thresholds = np.arange(0.01, 1.0, 0.01)
costs = []
fps = []
fns = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    if cm.shape == (2, 2):
        FP, FN = cm[0, 1], cm[1, 0]
    else:
        FP, FN = 0, 0
    costs.append((FP * COST_FP) + (FN * COST_FN))
    fps.append(FP)
    fns.append(FN)

min_cost = min(costs)
opt_idx = costs.index(min_cost)
opt_t = thresholds[opt_idx]
def_idx = (np.abs(thresholds - 0.5)).argmin()

# 1. Aşama: Sonuçları Terminale Yazdır
print("\n================ SİMÜLASYON SONUÇLARI ================")
print(f"Standart %50 Eşik Maliyeti: {costs[def_idx]:,.2f} TL (Kaçırılan Paket: {fns[def_idx]})")
print(f"Optimal %{opt_t*100:.0f} Eşik Maliyeti: {min_cost:,.2f} TL (Kaçırılan Paket: {fns[opt_idx]})")
print(f"------------------------------------------------------")
print(f" ŞİRKETE SAĞLANAN NET TASARRUF: {(costs[def_idx] - min_cost):,.2f} TL")
print("======================================================")

# 2. Aşama: Gizli Terminali Bypass Et ve Doğrudan Dosyaya Yazdır
with open('gercek_simulasyon_raporu.txt', 'w', encoding='utf-8') as f:
    f.write("================ GERÇEKÇİ KOBİ SİMÜLASYONU ================\n")
    f.write(f"Standart %50 Eşik Maliyeti: {costs[def_idx]:,.2f} TL (Kaçırılan Paket: {fns[def_idx]})\n")
    f.write(f"Optimal %{opt_t*100:.0f} Eşik Maliyeti: {min_cost:,.2f} TL (Kaçırılan Paket: {fns[opt_idx]})\n")
    f.write("------------------------------------------------------\n")
    f.write(f"NET TASARRUF: {(costs[def_idx] - min_cost):,.2f} TL\n")
    f.write("======================================================\n")

print("\nİşlem tamam! 'gercek_simulasyon_raporu.txt' dosyası proje klasöründe oluşturuldu.")

In [12]:
import shap
import matplotlib.pyplot as plt

print("SHAP (Açıklanabilir Yapay Zeka) Analizi Başlıyor...")

# XGBoost modeli için SHAP ağaç açıklayıcısını kur
explainer = shap.TreeExplainer(model)

# Test verisi üzerinden SHAP değerlerini hesapla
shap_values = explainer.shap_values(X_test)

# SHAP Özet Grafiğini (Summary Plot) Oluştur ve Kaydet
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Özeti: Modelin Karar Mekanizmasını Etkileyen Faktörler", fontsize=14)
plt.tight_layout()
plt.savefig('shap_ozeti.png')
print(" Harika! Grafik 'shap_ozeti.png' olarak projeye kaydedildi.")